In [ ]:
from pydicom.dataset import Dataset
from pynetdicom import AE, debug_logger
from pynetdicom.sop_class import ModalityPerformedProcedureStepRetrieve
from pydicom.dataset import FileDataset, FileMetaDataset
from pydicom.uid import UID
from pynetdicom.status import Status


def RIS_SYSTEM(inputdate):
    """
       input : date 
       
       ouput : return matched patient information 
       
    """
    debug_logger()
    ae = AE()
    ae.ae_title ="FAXITRON"    
    ae.add_requested_context(ModalityPerformedProcedureStepRetrieve)
    #ae.add_requested_context(ModalityWorklistInformationFind)
    #ae.add_requested_context(ModalityPerformedProcedureStep)

    # Create our Identifier (query) dataset
    ds = Dataset()
    ds.QueryRetrieveLevel = 'PATIENT'
    ds.ScheduledStationAETitle ='FILMDIGITIZE'   
    ds.ScheduledProcedureStepStartDate =inputdate   # DYNAMIC

    # Associate with the peer AE at IP 127.0.0.1 and port 11112
    assoc = ae.associate("192.168.1.199", 107,ae_title='DVTK')
    if assoc.is_established:
        # Send the C-FIND request
        
        responses = assoc.send_c_find(ds, ModalityPerformedProcedureStepRetrieve )
        print('DESIRED DATA ' , ds)
        for (status, identifier) in responses:
                        status2 = Dataset()
            status2.add_new(0x00000900, 'US', 65280)
            print(status2)
        
            if status == status2:
           
                print('DATA IN RIS ' , identifier)
                print('ScheduledStationAETitle', identifier[0x0040, 0x0100][0][0x0040, 0x0010].value)
                print('ScheduledProcedureStepStartDate' , identifier[0x0040, 0x0100][0][0x0040, 0x0002].value)
            
            
                if  identifier[0x0040, 0x0100][0][0x0040, 0x0010].value == ds.ScheduledStationAETitle:
                    print([])
                    if identifier[0x0040, 0x0100][0][0x0040, 0x0002].value == ds.ScheduledProcedureStepStartDate:
                        print(identifier)
                        matched_patientName = identifier[0x0010,0x0010].value
                        matched_patientID = identifier[0x0010,0x0020].value
                        matched_accessID = identifier[0x0008,0x0050].value
                        matched_date =identifier[0x0040, 0x0100][0][0x0040, 0x0002].value
                        matched_gender = identifier[0x0010,0x0040].value
                        
                        return ('information of desired data' ,matched_patientName,matched_patientID, matched_accessID ,matched_date,matched_gender )
                    
                    else:
                        return(' there is no patient with this features')
            elif status == 'Success':
                print('C-FIND finished, releasing the association')
            elif status == 'Cancel':
                print('C-FIND cancelled, releasing the association')
            elif status == 'Failure':
                print('C-FIND failed, releasing the association')

    # Release the association
        assoc.release()
    else:
        print('Association rejected, aborted or never connected')

        
inputdate = '20070126'
RIS_SYSTEM(inputdate)


I: Requesting Association
D: Request Parameters:
D: ======================= OUTGOING A-ASSOCIATE-RQ PDU ========================
D: Our Implementation Class UID:      1.2.826.0.1.3680043.9.3811.2.0.2
D: Our Implementation Version Name:   PYNETDICOM_202
D: Application Context Name:    1.2.840.10008.3.1.1.1
D: Calling Application Name:    KOEFAXITRON
D: Called Application Name:     DVTK
D: Our Max PDU Receive Size:    16382
D: Presentation Context:
D:   Context ID:        1 (Proposed)
D:     Abstract Syntax: =Modality Performed Procedure Step Retrieve SOP Class
D:     Proposed SCP/SCU Role: Default
D:     Proposed Transfer Syntaxes:
D:       =Implicit VR Little Endian
D:       =Explicit VR Little Endian
D:       =Deflated Explicit VR Little Endian
D:       =Explicit VR Big Endian
D: Requested Extended Negotiation: None
D: Requested Common Extended Negotiation: None
D: Requested Asynchronous Operations Window Negotiation: None
D: Requested User Identity Negotiation: None
D: ==============

DESIRED DATA  (0008, 0052) Query/Retrieve Level                CS: 'PATIENT'
(0040, 0001) Scheduled Station AE Title          AE: 'FILMDIGITIZE'
(0040, 0002) Scheduled Procedure Step Start Date DA: '20070126'


I:     (0032,1070) LO (no value available)                     # 0 RequestedContrastAgent
I:     (0040,0001) AE [FILMDIGITIZE]                           # 1 ScheduledStationAETitle
I:     (0040,0002) DA [20070126]                               # 1 ScheduledProcedureStepStartDate
I:     (0040,0003) TM [113500]                                 # 1 ScheduledProcedureStepStartTime
I:     (0040,0004) DA (no value available)                     # 0 ScheduledProcedureStepEndDate
I:     (0040,0005) TM (no value available)                     # 0 ScheduledProcedureStepEndTime
I:     (0040,0006) PN (no value available)                     # 0 ScheduledPerformingPhysicianName
I:     (0040,0007) LO [CSPINE]                                 # 1 ScheduledProcedureStepDescription
I: (0040,0008) SQ (Sequence with 1 item)                   # 1 ScheduledProtocolCodeSequence
I:       (Sequence item #1)
I:         (0008,0100) SH [CSPINE L BOTH]                          # 1 CodeValue
I:         (0008,0102) S

(0000, 0900) Status                              US: 65280
DATA IN RIS  (0008, 0005) Specific Character Set              CS: 'ISO_IR 100'
(0008, 0050) Accession Number                    SH: '00000187'
(0008, 0090) Referring Physician's Name          PN: 'Chief Radiologist^First^Middle^^'
(0008, 1110)  Referenced Study Sequence  1 item(s) ---- 
   (0008, 1150) Referenced SOP Class UID            UI: Detached Patient Management SOP Class
   (0008, 1155) Referenced SOP Instance UID         UI: 1.3.46.670589.16.12.2.1.176.53460.62132.20070126.102459.1
   ---------
(0010, 0010) Patient's Name                      PN: 'One^Secondary Capture Image'
(0010, 0020) Patient ID                          LO: 'pidP645'
(0010, 0021) Issuer of Patient ID                LO: 'QREmulator'
(0010, 0030) Patient's Birth Date                DA: '19800716'
(0010, 0040) Patient's Sex                       CS: 'M'
(0010, 1000) Other Patient IDs                   LO: ''
(0010, 1020) Patient's Size                

D: ========================== INCOMING DIMSE MESSAGE ==========================
D: Message Type                  : C-FIND RSP


('information of desired data',
 'One^Secondary Capture Image',
 'pidP645',
 '00000187',
 '20070126',
 'M')

D: Message ID Being Responded To : 1
D: Affected SOP Class UID        : Modality Performed Procedure Step Retrieve SOP Class
D: Identifier                    : None
D: Status                        : 0x0000
D: ============================ END DIMSE MESSAGE =============================
